In [ ]:
import pandas as pd
from transformers import pipeline
from transformers import AutoTokenizer, AutoModelForTokenClassification
import spacy
import negspacy
from spacy.tokens import Span
from negspacy.negation import Negex

In [ ]:
MODEL_NAME = "d4data/biomedical-ner-all"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME)

In [ ]:
def clean_corpus(corpus):
    cleaned_corpus = corpus.replace(".", " ")
    return cleaned_corpus

In [ ]:
import re
def extract_age(value):
    match = re.search(r"\d+", value)
    return int(match.group()) if match else None

In [ ]:
def aggregate_entities(df):
    result = {
        "Age": None,
        "Sex": None,
        "History": [],
        "Symptoms": [],
        "Medication": []
    }

    if df.empty:
        return pd.DataFrame([result])

    for group, subdf in df.groupby("entity_group"):
        values = []
        for _, row in subdf.iterrows():
            v = row["value"]
            if row.get("negated", False):
                v = f"{v} (negated)"
            values.append(v)

        if group == "Age":
            ages = [extract_age(v) for v in values if extract_age(v) is not None]
            result[group] = ages[0] if ages else None

        elif group == "Sex":
            result[group] = values[0] if values else None

        else:
            seen = set()
            unique_values = []
            for v in values:
                if v not in seen:
                    seen.add(v)
                    unique_values.append(v)

            result[group] = unique_values

    return pd.DataFrame([result])

In [ ]:
def detect_negations(corpus, df):
    if df.empty:
        return []

    nlp = spacy.load("en_core_web_sm")
    nlp.add_pipe("negex")

    doc = nlp(corpus)

    # Build spans (keep None for alignment)
    spans = [
        doc.char_span(
            int(row["start"]),
            int(row["end"]),
            label=row["entity_group"],
            alignment_mode="expand"
        )
        for _, row in df.iterrows()
    ]

    # Assign only valid spans
    doc.ents = [s for s in spans if s is not None]

    # Run negation AFTER setting entities
    doc = nlp.get_pipe("negex")(doc)

    # Extract negation flags, preserving order
    negated_flags = []
    ent_iter = iter(doc.ents)

    for span in spans:
        if span is None:
            negated_flags.append(False)
        else:
            negated_flags.append(next(ent_iter)._.negex)

    return negated_flags

In [ ]:
def ner_prediction(corpus):
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME)

    ner_pipeline = pipeline(
        "token-classification",
        model=model,
        tokenizer=tokenizer,
        aggregation_strategy="max",
    )

    cleaned_corpus = clean_corpus(corpus)

    pred = ner_pipeline(cleaned_corpus)

    pred_df = pd.DataFrame(pred)

    pred_df['negated'] = detect_negations(corpus, pred_df)
    
    actual_values_list = []
    for _, pred_df_row in pred_df.iterrows():
        actual_word = corpus[pred_df_row['start']: pred_df_row['end']]
        actual_values_list.append(actual_word)

    pred_df['value'] = actual_values_list
    
    if len(pred_df) != 0:
        pred_df = pred_df[['entity_group', 'value', 'word', 'start', 'end', 'score', 'negated']]
        pred_df['entity_group'] = pred_df['entity_group'].replace({"Sign_symptom": "Symptoms"})
        pred_df = pred_df.drop_duplicates(
            subset=['entity_group', 'value', 'negated'],
            keep='first'
        ).reset_index(drop=True)
        
    final_df = aggregate_entities(pred_df)
    final_df = final_df.reindex(columns=["Age", "Sex", "History", "Symptoms", "Medication"])

    return pred_df, final_df

In [ ]:
texto = (
    "The patient is a 65-year-old male with no history of hypertension and "
    "type 2 diabetes mellitus. He presented with chest pain and no shortness "
    "of breath. He was started on aspirin 81 mg daily and metoprolol 25 mg "
    "twice daily. CT scan revealed bilateral pulmonary embolism."
)

pred_df, final_df = ner_prediction(texto)
final_df

In [ ]:
nlp = spacy.load("en_core_web_sm")
nlp.add_pipe("negex")

doc = nlp("The patient has no chest pain")
for e in doc.ents:
    print(e.text, e._.negex)

In [ ]:
from spacy.tokens import Span

doc = nlp("The patient has no chest pain")
doc.ents = [Span(doc, 4, 6, label="SYMPTOM")]  # "chest pain"

doc = nlp.get_pipe("negex")(doc)

for ent in doc.ents:
    print(ent.text, ent._.negex)